### Illusion of Memory

Every API call to an LLM is stateless

The model does not remember the previous request unless the previous conversation is included as the current request

#### Note

To use Google Gemini SDK, we must use the base_url "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"

To use Gemini with OpenAI SDK, we must use the base_url "https://generativelanguage.googleapis.com/v1beta/openai/"

This is because, the OpenAI SDK, automatically appends /chats/completions

In [23]:
import os
import dotenv

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"

dotenv.load_dotenv() # this line reads environment variables from .env file and loads them into the process environment (virtual environment) 
gemini_api_key = os.getenv('GEMINI_API_KEY') # to get the environment variable (GEMINI_API_KEY) from the process environment to which the environment variables are already loaded

if not gemini_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file!")
elif not gemini_api_key.startswith("AQ."):
    print("An API key was found, but it doesn't start AQ.")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


### Approach 1

In [24]:
messages = [
    {"role" : "user", "parts" : [{"text" : "Hi, my name is Lalam!"}]}
]

In [25]:
from google import genai
from google.genai import types

client = genai.Client(api_key=gemini_api_key)
response = client.models.generate_content(
    model='gemini-3.1-flash-lite',
    contents=messages,
    config=types.GenerateContentConfig(
        system_instruction = "You are a helpful assistant!"
    )
)

In [26]:
print(response)

sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        text="Hi Lalam! It's a pleasure to meet you. How are you doing today? Is there anything I can help you with?",
        thought_signature=b"\x124\n2\x01\x11M2\x0f\xd4\xed\xdf}\x04s\x9apt\xbf\xb3\xb5\x83\x16\xb7\xbd\xf0\xed\x04\xc8\xd5&\xe2\x90\xd0\x8dF\xa6\x00\xb8\x9e^\xa2IC\xa5\xfa\xea\x8f\xc3\xe9P'#\x8f"
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-3.1-flash-lite' prompt_feedback=None response_id='RwlaarfgNYfljuMP47SC2AU' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=28,
  prompt_token_count=16,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=16
    ),
  ],
  total_token_count=44
) model_status=None automatic_function_calling_history=[] parsed=None


In [27]:
print(response.text)

Hi Lalam! It's a pleasure to meet you. How are you doing today? Is there anything I can help you with?


In [ ]:
messages = [
    {'role':'user', 'parts' : [{"text" : "What is my name?"}]}
]

response = client.models.generate_content(
    model='gemini-3.1-flash-lite',
    contents=messages,
    config=types.GenerateContentConfig(
        system_instruction = "You are a helpful assistant!"
    )
)

print(response.text)

## The output is "I don't know" because the API call does not have any memory. It is completely stateless

I don’t know your name! As an AI, I don’t have access to your personal information or identity unless you have shared it with me in our current conversation. 

If you'd like me to know it, feel free to tell me!


In [ ]:
# to create an illusion of model having a memory, we need to include the previous conversation along with the current request

messages = [
    {'role' : 'user', 'parts' : [{'text' : 'Hello, my name is Lalam!'}]},
    {'role' : 'user', 'parts' : [{'text' : 'What is my name?'}]}
]

response = client.models.generate_content(
    model='gemini-3.1-flash-lite',
    contents=messages,
    config=types.GenerateContentConfig(
        system_instruction = 'You are a helpful assistant!'
    )
)

print(response.text)

Hello Lalam! Your name is Lalam. It's nice to meet you!


## Approach 2

### Illustration of Memory using OpenAI compatible API

In [30]:
from openai import OpenAI

gemini_openai_compatible_base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

client = OpenAI(api_key=gemini_api_key, base_url=gemini_openai_compatible_base_url)

messages = [
    {'role':'system', 'content':'You are a friendly and helpful assistant!'},
    {'role':'user', 'content':'Hello, my. name is Ramu!'}
]

response = client.chat.completions.create(
    model = 'gemini-3.1-flash-lite',
    messages = messages
)

print(response.choices[0].message.content)

Hello, Ramu! It’s very nice to meet you. How are you doing today? Is there anything I can help you with?


In [ ]:
messages = [
    {'role':'system', 'content':'You are a helpful assistant'},
    {'role':'user', 'content':'What is my name?'}
]

response = client.chat.completions.create(
    model='gemini-3.1-flash-lite',
    messages=messages
)

print(response.choices[0].message.content)

## The output is "I don't know" because the API call does not have any memory. It is completely stateless

I don't know your name. As an AI, I don't have access to your personal information or identity unless you have shared it with me in our current conversation. 

If you'd like, you can tell me your name, and I will remember it for the rest of our chat!


In [ ]:
messages = [
    {'role':'system', 'content':'You are a helpful assistant'},
    {'role':'user', 'content':'Hello, my. name is Ramu!'},
    {'role':'user', 'content':'What is my name?'}
]

response = client.chat.completions.create(
    model = 'gemini-3.1-flash-lite',
    messages = messages
)

print(response.choices[0].message.content)

# to create an illusion of model having a memory, we need to include the previous conversation along with the current request 

Hello Ramu! Your name is Ramu. It's nice to meet you!
